# ML Enhancement: In-Store Journey Analysis

Analyzes customer shopping paths using BLE beacon zone transitions to understand in-store behavior patterns.

## Data Flow
```
Silver (configured journey source tables) --> Gold (configured journey analysis outputs)
```

## Business Value
- Optimize store layout based on traffic patterns
- Understand customer engagement by zone
- Identify high-value path sequences
- Correlate paths with purchase outcomes

## Outputs
- **Journey patterns output**: Top N common paths with conversion/basket metrics
- **Zone transitions output**: Zone transition probability matrix
- **Zone dwell stats output**: Average dwell time statistics by zone

## Usage
Run this notebook **on-demand** or schedule periodically (e.g., daily) to update journey insights.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType, ArrayType
from pyspark.sql.utils import AnalysisException
from datetime import datetime, timezone, timedelta
import os
import mlflow


In [ ]:
# =============================================================================
# PARAMETERS
# =============================================================================

def get_env(var_name, default=None):
    return os.environ.get(var_name, default)

LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", default="retail_lakehouse")
SILVER_DB = get_env("SILVER_DB", default="silver")
GOLD_DB = get_env("GOLD_DB", default="gold")
EXPERIMENT_NAME = get_env("MLFLOW_EXPERIMENT", default="journey_analysis")
ZONE_CHANGES_TABLE = get_env("ZONE_CHANGES_TABLE", default="fact_customer_zone_changes")
RECEIPTS_TABLE = get_env("RECEIPTS_TABLE", default="fact_receipts")
JOURNEY_PATTERNS_TABLE = get_env("JOURNEY_PATTERNS_TABLE", default="journey_patterns")
ZONE_TRANSITIONS_TABLE = get_env("ZONE_TRANSITIONS_TABLE", default="zone_transitions")
ZONE_DWELL_STATS_TABLE = get_env("ZONE_DWELL_STATS_TABLE", default="zone_dwell_stats")
JOURNEY_PATTERNS_TABLE_NAME = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{JOURNEY_PATTERNS_TABLE}"
ZONE_TRANSITIONS_TABLE_NAME = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{ZONE_TRANSITIONS_TABLE}"
ZONE_DWELL_STATS_TABLE_NAME = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{ZONE_DWELL_STATS_TABLE}"

# Analysis window: last 30 days by default
ANALYSIS_DAYS = int(get_env("ANALYSIS_DAYS", default="30"))
MIN_PATH_LENGTH = int(get_env("MIN_PATH_LENGTH", default="2"))
TOP_N_PATHS = int(get_env("TOP_N_PATHS", default="20"))
SESSION_TIMEOUT_MINUTES = int(get_env("SESSION_TIMEOUT_MINUTES", default="30"))
DWELL_OUTLIER_CUTOFF_SECONDS = int(get_env("DWELL_OUTLIER_CUTOFF_SECONDS", default="3600"))
RECEIPT_MATCH_GRACE_WINDOW_MINUTES = int(get_env("RECEIPT_MATCH_GRACE_WINDOW_MINUTES", default="5"))



print(f"Configuration:")
print(f"  SILVER_DB={SILVER_DB}")
print(f"  GOLD_DB={GOLD_DB}")
print(f"  Source tables: {ZONE_CHANGES_TABLE}, {RECEIPTS_TABLE}")
print(f"  Output tables: {JOURNEY_PATTERNS_TABLE_NAME}, {ZONE_TRANSITIONS_TABLE_NAME}, {ZONE_DWELL_STATS_TABLE_NAME}")
print(f"  ANALYSIS_DAYS={ANALYSIS_DAYS}")
print(f"  MIN_PATH_LENGTH={MIN_PATH_LENGTH}")
print(f"  TOP_N_PATHS={TOP_N_PATHS}")
print(f"  SESSION_TIMEOUT_MINUTES={SESSION_TIMEOUT_MINUTES}")
print(f"  DWELL_OUTLIER_CUTOFF_SECONDS={DWELL_OUTLIER_CUTOFF_SECONDS}")
print(f"  RECEIPT_MATCH_GRACE_WINDOW_MINUTES={RECEIPT_MATCH_GRACE_WINDOW_MINUTES}")

In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def ensure_database(name):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {LAKEHOUSE_NAME}.{name}")

def read_silver(table_name):
    return spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")

def save_gold(df, table_name):
    full_name = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{table_name}"
    df.write.format("delta").mode("overwrite").saveAsTable(full_name)
    print(f"  {full_name}: {df.count()} rows")

def silver_exists(table_name):
    try:
        spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")
        return True
    except AnalysisException:
        return False

def resolve_table_column(df, table_name, *candidates):
    columns_by_lower = {col.lower(): col for col in df.columns}
    for candidate in candidates:
        resolved = columns_by_lower.get(candidate.lower())
        if resolved is not None:
            return resolved
    raise RuntimeError(
        f"Unable to resolve any of {candidates} in {LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}. Available columns: {df.columns}"
    )

ensure_database(GOLD_DB)

mlflow.set_experiment(EXPERIMENT_NAME)


In [ ]:
print("="*60)
print("IN-STORE JOURNEY ANALYSIS")
print("="*60)

## Step 1: Reconstruct Customer Paths

Build sequential paths from `customer_zone_changed` events, grouping by customer session.

In [ ]:
if not silver_exists(ZONE_CHANGES_TABLE):
    raise ValueError(f"{ZONE_CHANGES_TABLE} table not found in Silver layer")

df_zone_changes_raw = read_silver(ZONE_CHANGES_TABLE)
zone_store_id_col = resolve_table_column(df_zone_changes_raw, ZONE_CHANGES_TABLE, "store_id", "StoreID")
zone_customer_ble_col = resolve_table_column(df_zone_changes_raw, ZONE_CHANGES_TABLE, "customer_ble_id", "CustomerBLEId")
zone_from_col = resolve_table_column(df_zone_changes_raw, ZONE_CHANGES_TABLE, "from_zone", "FromZone")
zone_to_col = resolve_table_column(df_zone_changes_raw, ZONE_CHANGES_TABLE, "to_zone", "ToZone")
zone_event_ts_col = resolve_table_column(df_zone_changes_raw, ZONE_CHANGES_TABLE, "event_ts", "EventTS")

# Materialize zone change source projections with Spark timestamps/dates for Fabric runtime safety
df_zone_changes_source = (
    df_zone_changes_raw
    .select(
        F.col(zone_store_id_col).alias("store_id"),
        F.col(zone_customer_ble_col).alias("customer_ble_id"),
        F.col(zone_from_col).alias("from_zone"),
        F.col(zone_to_col).alias("to_zone"),
        F.col(zone_event_ts_col).cast("timestamp").alias("event_ts")
    )
    .withColumn("event_date", F.to_date("event_ts"))
    .cache()
)

zone_source_count = df_zone_changes_source.count()
latest_date_row = df_zone_changes_source.agg(F.max("event_date").alias("latest_date")).first()
latest_date_value = latest_date_row["latest_date"]

if latest_date_value is None:
    raise ValueError(f"{ZONE_CHANGES_TABLE} contains no event_ts values")

analysis_start_date_value = latest_date_value - timedelta(days=ANALYSIS_DAYS)
latest_date_literal = F.lit(latest_date_value).cast("date")
analysis_start_date = F.lit(analysis_start_date_value).cast("date")

df_zone_changes = (
    df_zone_changes_source
    .filter(
        (F.col("event_date") >= analysis_start_date) &
        (F.col("event_date") <= latest_date_literal)
    )
    .select(
        "store_id",
        "customer_ble_id",
        "from_zone",
        "to_zone",
        "event_ts",
        "event_date"
    )
)

zone_change_count = df_zone_changes.count()
print(f"Materialized {zone_source_count} zone change source rows")
print(f"Loaded {zone_change_count} zone change events from {analysis_start_date_value} to {latest_date_value}")

In [ ]:
# Define sessions: group events by customer and store using the configured session timeout
# Session ends if there's a gap greater than SESSION_TIMEOUT_MINUTES between events

window_by_customer = Window.partitionBy("store_id", "customer_ble_id").orderBy("event_ts")

df_with_sessions = (
    df_zone_changes
    .withColumn(
        "prev_event_ts",
        F.lag("event_ts").over(window_by_customer)
    )
    .withColumn(
        "time_gap_minutes",
        (F.unix_timestamp("event_ts") - F.unix_timestamp("prev_event_ts")) / 60
    )
    .withColumn(
        "is_new_session",
        F.when(
            (F.col("prev_event_ts").isNull()) | (F.col("time_gap_minutes") > SESSION_TIMEOUT_MINUTES),
            1
        ).otherwise(0)
    )
    .withColumn(
        "session_id",
        F.sum("is_new_session").over(
            window_by_customer.rowsBetween(Window.unboundedPreceding, Window.currentRow)
        )
    )
    .withColumn(
        "session_key",
        F.concat(
            F.col("store_id").cast("string"),
            F.lit("_"),
            F.col("customer_ble_id"),
            F.lit("_"),
            F.col("session_id").cast("string")
        )
    )
)

print(f"Identified {df_with_sessions.select('session_key').distinct().count()} unique customer sessions")

In [ ]:
# Build paths: collect deterministically ordered transitions for each session
df_paths = (
    df_with_sessions
    .groupBy("session_key", "store_id", "customer_ble_id")
    .agg(
        F.sort_array(
            F.collect_list(
                F.struct(
                    F.col("event_ts"),
                    F.col("from_zone"),
                    F.col("to_zone")
                )
            )
        ).alias("ordered_transitions"),
        F.min("event_ts").alias("session_start"),
        F.max("event_ts").alias("session_end")
    )
    .withColumn(
        "zone_path",
        F.expr("concat(array(element_at(ordered_transitions, 1).from_zone), transform(ordered_transitions, x -> x.to_zone))")
    )
    .withColumn(
        "path_length",
        F.size("zone_path")
    )
    .withColumn(
        "path_string",
        F.array_join("zone_path", " -> ")
    )
    .withColumn(
        "session_duration_minutes",
        (F.unix_timestamp("session_end") - F.unix_timestamp("session_start")) / 60
    )
    .filter(F.col("path_length") >= MIN_PATH_LENGTH)
)

print(f"Built {df_paths.count()} customer paths (min length: {MIN_PATH_LENGTH} zones)")

## Step 2: Calculate Zone Dwell Times

Compute average time spent in each zone during customer sessions.

In [ ]:
# Calculate dwell time per zone by looking at time between zone transitions
df_dwell_calc = (
    df_with_sessions
    .withColumn(
        "next_event_ts",
        F.lead("event_ts").over(
            Window.partitionBy("session_key").orderBy("event_ts")
        )
    )
    .withColumn(
        "dwell_seconds",
        F.when(
            F.col("next_event_ts").isNotNull(),
            F.unix_timestamp("next_event_ts") - F.unix_timestamp("event_ts")
        ).otherwise(None)
    )
    .filter(F.col("dwell_seconds").isNotNull())
    .filter(F.col("dwell_seconds") > 0)
    .filter(F.col("dwell_seconds") < DWELL_OUTLIER_CUTOFF_SECONDS)  # Filter dwell-time outliers
)

df_zone_dwell_stats = (
    df_dwell_calc
    .groupBy("store_id", "to_zone")
    .agg(
        F.avg("dwell_seconds").alias("avg_dwell_seconds"),
        F.expr("percentile_approx(dwell_seconds, 0.5)").alias("median_dwell_seconds"),
        F.min("dwell_seconds").alias("min_dwell_seconds"),
        F.max("dwell_seconds").alias("max_dwell_seconds"),
        F.count("*").alias("visit_count")
    )
    .withColumnRenamed("to_zone", "zone")
    .withColumn("computed_at", F.current_timestamp())
)

print(f"Creating {ZONE_DWELL_STATS_TABLE_NAME}...")
save_gold(df_zone_dwell_stats, ZONE_DWELL_STATS_TABLE)

## Step 3: Calculate Zone Transition Probabilities

Build a transition matrix showing the probability of moving from one zone to another.

In [ ]:
# Count transitions between zones
df_transitions = (
    df_zone_changes
    .groupBy("store_id", "from_zone", "to_zone")
    .agg(F.count("*").alias("transition_count"))
)

# Calculate total transitions from each zone
df_from_totals = (
    df_transitions
    .groupBy("store_id", "from_zone")
    .agg(F.sum("transition_count").alias("total_from_zone"))
)

# Join and calculate probabilities
df_zone_transitions = (
    df_transitions
    .join(
        df_from_totals,
        on=["store_id", "from_zone"],
        how="inner"
    )
    .withColumn(
        "transition_probability",
        F.col("transition_count") / F.col("total_from_zone")
    )
    .withColumn("computed_at", F.current_timestamp())
    .select(
        "store_id",
        "from_zone",
        "to_zone",
        "transition_count",
        "transition_probability",
        "computed_at"
    )
    .orderBy("store_id", "from_zone", F.desc("transition_probability"))
)

print(f"Creating {ZONE_TRANSITIONS_TABLE_NAME}...")
save_gold(df_zone_transitions, ZONE_TRANSITIONS_TABLE)

## Step 4: Identify Common Path Patterns

Find the top N most frequent customer paths through the store.

In [ ]:
# Count path frequency
df_path_frequency = (
    df_paths
    .groupBy("store_id", "path_string", "path_length")
    .agg(
        F.count("*").alias("occurrence_count"),
        F.avg("session_duration_minutes").alias("avg_session_duration_minutes")
    )
)

# Rank paths by frequency per store
window_by_store = Window.partitionBy("store_id").orderBy(F.desc("occurrence_count"))

df_top_paths = (
    df_path_frequency
    .withColumn("rank", F.row_number().over(window_by_store))
    .filter(F.col("rank") <= TOP_N_PATHS)
)

print(f"Identified top {TOP_N_PATHS} paths per store")

## Step 5: Correlate Paths with Purchase Outcomes

Join customer paths with receipt data to understand conversion and basket size by journey.

In [ ]:
# Check if receipts are available
if silver_exists(RECEIPTS_TABLE):
    df_receipts_raw = read_silver(RECEIPTS_TABLE)
    receipt_store_id_col = resolve_table_column(df_receipts_raw, RECEIPTS_TABLE, "store_id", "StoreID")
    receipt_event_ts_col = resolve_table_column(df_receipts_raw, RECEIPTS_TABLE, "event_ts", "EventTS")
    receipt_total_cents_col = resolve_table_column(df_receipts_raw, RECEIPTS_TABLE, "total_cents", "TotalCents")
    receipt_id_ext_col = resolve_table_column(df_receipts_raw, RECEIPTS_TABLE, "receipt_id_ext", "ReceiptIdExt", "ReceiptID", "ReceiptId")

    # Materialize receipt source projections with Spark timestamps/dates for Fabric runtime safety
    df_receipts_source = (
        df_receipts_raw
        .select(
            F.col(receipt_store_id_col).alias("store_id"),
            F.col(receipt_event_ts_col).cast("timestamp").alias("event_ts"),
            F.col(receipt_total_cents_col).alias("total_cents"),
            F.col(receipt_id_ext_col).alias("receipt_id_ext")
        )
        .withColumn("event_date", F.to_date("event_ts"))
        .cache()
    )

    receipts_source_count = df_receipts_source.count()
    df_receipts = (
        df_receipts_source
        .filter(
            (F.col("event_date") >= analysis_start_date) &
            (F.col("event_date") <= latest_date_literal)
        )
        .withColumn("receipt_event_epoch", F.unix_timestamp("event_ts"))
        .select(
            "store_id",
            "event_ts",
            "event_date",
            "receipt_event_epoch",
            "total_cents",
            "receipt_id_ext"
        )
    )
    print(f"Materialized {receipts_source_count} receipt source rows")

    # Customer identifiers may not align across sources, so correlate by store + time window.
    # A purchase is matched when it occurs in the same store within the session timeframe.
    grace_window_seconds = RECEIPT_MATCH_GRACE_WINDOW_MINUTES * 60

    df_sessions_for_receipts = (
        df_paths
        .select(
            "store_id",
            "session_key",
            "path_string",
            "path_length",
            "session_duration_minutes",
            "session_start",
            "session_end"
        )
        .withColumn("session_start_epoch", F.unix_timestamp("session_start"))
        .withColumn("session_end_epoch", F.unix_timestamp("session_end"))
    )

    # Create session-to-receipt mapping by joining on store and epoch-second proximity
    df_path_receipts = (
        df_sessions_for_receipts.alias("paths")
        .join(
            df_receipts.alias("receipts"),
            (
                (F.col("paths.store_id") == F.col("receipts.store_id")) &
                (F.col("receipts.receipt_event_epoch") >= F.col("paths.session_start_epoch")) &
                (F.col("receipts.receipt_event_epoch") <= (F.col("paths.session_end_epoch") + F.lit(grace_window_seconds)))
            ),
            how="left"
        )
        .select(
            F.col("paths.store_id").alias("store_id"),
            F.col("paths.session_key").alias("session_key"),
            F.col("paths.path_string").alias("path_string"),
            F.col("paths.path_length").alias("path_length"),
            F.col("paths.session_duration_minutes").alias("session_duration_minutes"),
            F.col("receipts.receipt_id_ext").alias("receipt_id_ext"),
            F.col("receipts.total_cents").alias("total_cents")
        )
    )

    # Aggregate metrics at the session level first to avoid inflation from multiple receipt matches
    df_session_metrics = (
        df_path_receipts
        .groupBy("store_id", "session_key", "path_string", "path_length")
        .agg(
            F.first("session_duration_minutes").alias("session_duration_minutes"),
            F.max(
                F.when(F.col("receipt_id_ext").isNotNull(), F.lit(1)).otherwise(F.lit(0))
            ).alias("has_purchase"),
            F.sum(
                F.when(F.col("total_cents").isNotNull(), F.col("total_cents")).otherwise(F.lit(0))
            ).alias("session_basket_cents")
        )
    )

    # Aggregate session metrics to the path level
    df_path_metrics = (
        df_session_metrics
        .groupBy("store_id", "path_string", "path_length")
        .agg(
            F.count("*").alias("total_sessions"),
            F.sum("has_purchase").alias("sessions_with_purchase"),
            F.avg("session_duration_minutes").alias("avg_session_duration_minutes"),
            F.avg(
                F.when(F.col("has_purchase") == 1, F.col("session_basket_cents")).otherwise(None)
            ).alias("avg_basket_cents"),
            F.sum("session_basket_cents").alias("total_revenue_cents")
        )
        .withColumn(
            "conversion_rate",
            F.when(
                F.col("total_sessions") > 0,
                F.col("sessions_with_purchase") / F.col("total_sessions")
            ).otherwise(F.lit(None).cast("double"))
        )
    )

    # Join with top paths
    df_journey_patterns = (
        df_top_paths.alias("top")
        .join(
            df_path_metrics.alias("metrics"),
            on=["store_id", "path_string", "path_length"],
            how="left"
        )
        .withColumn("computed_at", F.current_timestamp())
        .withColumn("analysis_period_days", F.lit(ANALYSIS_DAYS))
        .select(
            F.col("store_id"),
            F.col("top.rank").alias("rank"),
            F.col("path_string"),
            F.col("path_length"),
            F.col("top.occurrence_count").alias("occurrence_count"),
            F.coalesce(F.col("metrics.total_sessions"), F.col("top.occurrence_count")).alias("total_sessions"),
            F.col("metrics.sessions_with_purchase").alias("sessions_with_purchase"),
            F.col("metrics.conversion_rate").alias("conversion_rate"),
            F.coalesce(F.col("metrics.avg_session_duration_minutes"), F.col("top.avg_session_duration_minutes")).alias("avg_session_duration_minutes"),
            F.col("metrics.avg_basket_cents").alias("avg_basket_cents"),
            F.col("metrics.total_revenue_cents").alias("total_revenue_cents"),
            F.col("analysis_period_days"),
            F.col("computed_at")
        )
        .orderBy("store_id", "rank")
    )
    
else:
    # No receipts available, just output path frequency
    print(f"Warning: {RECEIPTS_TABLE} not found, skipping conversion analysis")
    df_journey_patterns = (
        df_top_paths
        .withColumn("total_sessions", F.col("occurrence_count"))
        .withColumn("sessions_with_purchase", F.lit(None).cast("long"))
        .withColumn("conversion_rate", F.lit(None).cast("double"))
        .withColumn("avg_basket_cents", F.lit(None).cast("double"))
        .withColumn("total_revenue_cents", F.lit(None).cast("long"))
        .withColumn("computed_at", F.current_timestamp())
        .withColumn("analysis_period_days", F.lit(ANALYSIS_DAYS))
        .select(
            "store_id",
            "rank",
            "path_string",
            "path_length",
            "occurrence_count",
            "total_sessions",
            "sessions_with_purchase",
            "conversion_rate",
            "avg_session_duration_minutes",
            "avg_basket_cents",
            "total_revenue_cents",
            "analysis_period_days",
            "computed_at"
        )
        .orderBy("store_id", "rank")
    )

print(f"Creating {JOURNEY_PATTERNS_TABLE_NAME}...")
save_gold(df_journey_patterns, JOURNEY_PATTERNS_TABLE)

## Summary

Display key insights from the journey analysis.

In [ ]:
# Log results to MLflow
with mlflow.start_run(
    run_name=f"journey_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M')}"
) as run:
    mlflow.log_params({
        "algorithm": "path_analysis",
        "analysis_days": ANALYSIS_DAYS,
        "min_path_length": MIN_PATH_LENGTH,
        "top_n_paths": TOP_N_PATHS,
        "session_timeout_minutes": SESSION_TIMEOUT_MINUTES,
        "dwell_outlier_cutoff_seconds": DWELL_OUTLIER_CUTOFF_SECONDS,
        "receipt_match_grace_window_minutes": RECEIPT_MATCH_GRACE_WINDOW_MINUTES,
    })

    patterns_count = spark.table(JOURNEY_PATTERNS_TABLE_NAME).count()
    transitions_count = spark.table(ZONE_TRANSITIONS_TABLE_NAME).count()
    dwell_count = spark.table(ZONE_DWELL_STATS_TABLE_NAME).count()
    mlflow.log_metrics({
        "journey_patterns": patterns_count,
        "zone_transitions": transitions_count,
        "dwell_stats": dwell_count,
    })

    print(f"MLflow run: {run.info.run_id}")
    print(f"Patterns: {patterns_count}, Transitions: {transitions_count}, Dwell: {dwell_count}")


In [ ]:
print("\n" + "="*60)
print("JOURNEY ANALYSIS COMPLETE")
print("="*60)

print(f"\nOutput Tables:")
print(f"  {JOURNEY_PATTERNS_TABLE_NAME} - Top {TOP_N_PATHS} paths per store with conversion metrics")
print(f"  {ZONE_TRANSITIONS_TABLE_NAME} - Zone transition probability matrix")
print(f"  {ZONE_DWELL_STATS_TABLE_NAME} - Zone dwell time statistics")

print(f"\nSample Results (Top 5 Paths):")
df_journey_patterns.limit(5).show(truncate=False)

print(f"\nAnalysis Period: Last {ANALYSIS_DAYS} days")
print(f"Computed At: {datetime.now(timezone.utc)}")